<h1 align="center">
  <span style="background: linear-gradient(90deg, rgb(163, 99, 236), rgb(125, 43, 100)); 
               -webkit-background-clip: text; 
               -webkit-text-fill-color: transparent;">
    SOLAR FLARE DATASET
  </span>
</h1>


---

Cada fila de este dataset corresponde a una región del sol, las columnas pues la característica de la región, el objetivo es predecir si habrá una llamarada solar
ej -> H A X 1 3 1 1 1 1 1 0 0 0

los 0 0 0 -> corresponde a las clases objetivo , si habra flare o no 
son common flares, moderate flares, severe flares ; son cantidades 

cada fila de esas nos indicara cuantas llamaradas de cada tipo habrá 

0 0 0 -> no habrá flares 

2 1 0 -> 2 comunes 1 moderada 0 severas 

Este problema es de tipo regresión multivariable (multi-outpùt regression)



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd

import polars as pl
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter  
from torch.utils.data import random_split
from torch.utils.data import Dataset
from random import sample



In [2]:
class StandardScaler:
    def __init__(self, mean=None, std=None, epsilon=1e-7):
        self.mean = mean
        self.std = std
        self.epsilon = epsilon

    def fit(self, values):
        dims = list(range(values.dim() - 1))
        self.mean = torch.mean(values, dim=dims)
        self.std = torch.std(values, dim=dims)

    def transform(self, values):
        return (values - self.mean) / (self.std + self.epsilon)

    def fit_transform(self, values):
        self.fit(values)
        return self.transform(values)

    def __call__(self, sample):
        values, saidas = sample
        # Escalamos solo las características (values), no las salidas (saidas)
        return ((values - self.mean) / (self.std + self.epsilon), saidas)
        
    def __repr__(self):
        return f"mean: {self.mean}, std:{self.std}, epsilon:{self.epsilon}"

Cargamos los datos con lazy polars -> no se ejecutan las operaciones inmediatamente , mira la lista de tareas y toma la opcion más óptima 

In [3]:
class SolarFlareDataset(Dataset):
    def __init__(self, file1, file2, transform=None):
        self.transform = transform
        column_names = [
            "class", "large_spot_size", "spot_distribution", "activity", 
            "evolution", "prev_24h_flare_activity", "hist_complex", 
            "become_complex", "area", "area_large_spot",
            "common_flares", "moderate_flares", "severe_flares"
        ]

        # mapeamos las letras a numeros para que pytorch pueda trabajar con ellas
        mapping_exp = [
            pl.col("class").cast(pl.Categorical).to_physical().cast(pl.Float32),
            pl.col("large_spot_size").cast(pl.Categorical).to_physical().cast(pl.Float32),
            pl.col("spot_distribution").cast(pl.Categorical).to_physical().cast(pl.Float32),
        ]

        # TODO para leer en formato lazy acordarse de usar el scan_csv en vez del read_csv, y luego hacer un collect para traerlo a memoria!
        lazy1 = pl.scan_csv(file1, has_header=False, separator=" ", skip_rows=1, new_columns=column_names)
        lazy2 = pl.scan_csv(file2, has_header=False, separator=" ", skip_rows=1, new_columns=column_names)
        
        self.dataSet = pl.concat([lazy1, lazy2]).with_columns(mapping_exp).drop_nulls().with_row_index("id")

    def __len__(self):
        return self.dataSet.select(pl.len()).collect().item()
    
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        else: 
            idx = [idx]
        
        # filtramos el dataset para quedarnos solo con las filas que corresponden a los indices que nos han dado, y luego lo convertimos a un dataframe de pandas para poder trabajar con el
        seccion = self.dataSet.filter(pl.col("id").is_in(idx))
        datos = seccion.collect()

        preds_cols = datos.columns[1:11] # quitamos la de id 
        # el squeeze es para eliminar la dimension de 1 que queda al convertir a tensor, y asi tener un tensor de 10 elementos en vez de 10x1
        preds = torch.tensor(datos.select(preds_cols).to_numpy(), dtype=torch.float32).squeeze()

        target_cols = ["common_flares", "moderate_flares", "severe_flares"]
        # spcs es el tensor de las salidas, que son las columnas de common_flares, moderate_flares y severe_flares 
        spcs = torch.tensor(datos.select(target_cols).to_numpy(), dtype=torch.float32).squeeze()

        sample = (preds, spcs)

        if self.transform:
            sample = self.transform(sample)
        
        return sample


In [4]:
dataset = SolarFlareDataset("../../docs/solar-flare/flare.data1", "../../docs/solar-flare/flare.data2")
full_data = dataset.dataSet.collect()
cols_x = full_data.columns[1:11] # quitamos la de id

# calculamos la media y la desviacion estandar de cada columna para luego poder escalar los datos
all_x = torch.tensor(full_data.select(cols_x).to_numpy(), dtype=torch.float32)
media = torch.mean(all_x, dim=0)
std = torch.std(all_x, dim=0)

scaler = StandardScaler(mean=media, std=std)
final_dataset = SolarFlareDataset("../../docs/solar-flare/flare.data1", "../../docs/solar-flare/flare.data2", transform=scaler)

print(dataset[0])


(tensor([ 3., 11.,  0.,  1.,  2.,  1.,  1.,  2.,  1.,  2.]), tensor([0., 0., 0.]))


Creacion del modelo Pytorch

In [5]:
class RegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(RegressionModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 32)
        self.layer2 = nn.Linear(32, 16)
        self.layer3 = nn.Linear(16, 3)  # 3 salidas para common_flares, moderate_flares y severe_flares

    def forward(self, x):
        # F -> functional, es una forma de usar las funciones de activacion sin tener que definirlas como capas en el init, lo cual es util para funciones de activacion que no tienen parametros, como relu o sigmoid
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x) # salida lineal directa para regresión
    

acordase de cuando metes dependencias, reiniciar el kernel 

In [ ]:
model = RegressionModel(input_dim=10)  # 10 características de entrada
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

loss_fn = nn.MSELoss()
compiled_model = torch.compile(model)
print(compiled_model)

OptimizedModule(
  (_orig_mod): RegressionModel(
    (layer1): Linear(in_features=10, out_features=32, bias=True)
    (layer2): Linear(in_features=32, out_features=16, bias=True)
    (layer3): Linear(in_features=16, out_features=3, bias=True)
  )
)


DATALOADERS-> convierte el dataset como en batchs separados para que los procese de esta la memoria RAM no explota xd y al red neuronal puede ir ajustandose poco a poco 

In [ ]:
from torch.utils.data import random_split, DataLoader

lonxitudeDataset = len(final_dataset)
tamTrain = int(lonxitudeDataset * 0.8)
tamVal = lonxitudeDataset - tamTrain

print(f"Total datos: {lonxitudeDataset} | Train: {tamTrain} | Val: {tamVal}")

train_set, val_set = random_split(final_dataset, [tamTrain, tamVal])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

Total datos: 1389 | Train: 1111 | Val: 278


Entrenamiento: 

EPOCH -> lectura completa al dataset 
ENTOCNES si tu le pones = 30 lo leera 30 veces
en cada vuelta epoch la red va entendiendo mejor los patrones , obvimaente si le pnes 1 los suspenda                                                                                                                                                              
loss -> cant errores , bsucamos el < num



In [9]:
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for i, data in enumerate(train_loader):
        inputs, labels = data
        
        optimizer.zero_grad() # Limpiar gradientes
        
        outputs = model(inputs) # Predecir
        loss = loss_fn(outputs, labels) # Calcular error
        
        loss.backward() # Propagar el error hacia atrás
        optimizer.step() # Actualizar pesos
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    
    # a aptir de aqui entra la fase de validacion 
    model.eval()
    running_vloss = 0.0
    with torch.no_grad():
        for vinputs, vlabels in val_loader:
            voutputs = model(vinputs)
            vloss = loss_fn(voutputs, vlabels)
            running_vloss += vloss.item()
            
    avg_val_loss = running_vloss / len(val_loader)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"EPOCH {epoch+1:2d} | Train Loss (MSE): {avg_train_loss:.4f} | Val Loss (MSE): {avg_val_loss:.4f}")

EPOCH  1 | Train Loss (MSE): 0.2058 | Val Loss (MSE): 0.3267
EPOCH  5 | Train Loss (MSE): 0.1818 | Val Loss (MSE): 0.3088
EPOCH 10 | Train Loss (MSE): 0.1774 | Val Loss (MSE): 0.3109
EPOCH 15 | Train Loss (MSE): 0.1767 | Val Loss (MSE): 0.3128
EPOCH 20 | Train Loss (MSE): 0.1750 | Val Loss (MSE): 0.3171
EPOCH 25 | Train Loss (MSE): 0.1733 | Val Loss (MSE): 0.3204
EPOCH 30 | Train Loss (MSE): 0.1702 | Val Loss (MSE): 0.3230
